In [48]:
import argparse
import logging
import os
import sys

import pandas as pd
from sqlalchemy import create_engine, text


In [49]:
DATA_DIR = os.environ.get("WASEET_DATA_DIR", "../data")
DB_URL = os.environ.get("WASEET_DB_URL",
                        "postgresql+psycopg2://de:de@localhost:5442/waseet")

# TODO. The contract: the columns a scan file must have for this run to be worth
# starting. One of the twenty-one files renames one of them.
REQUIRED = ["scan_id", "parcel_id", "hub_id", "scan_type", "scanned_at", "weight_kg"]
COLUMNS = ["scan_id", "parcel_id", "hub_id", "scan_type", "scanned_at", "weight_kg", "courier_id", "customer_id", "service_level_id"]

In [50]:
os.makedirs("logs", exist_ok=True)
os.makedirs("quarantine", exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[logging.FileHandler("logs/pipeline.log"), logging.StreamHandler()])
log = logging.getLogger("waseet")

engine = create_engine(DB_URL)

In [51]:
def extract(scan_date):
    path = os.path.join(DATA_DIR, f"scans_{scan_date}.csv")
    df = pd.read_csv(path, dtype=str)
    
    if "timestamp" in df.columns and "scanned_at" not in df.columns:
        df = df.rename(columns={"timestamp": "scanned_at"})
    if "id" in df.columns and "scan_id" not in df.columns:
        df = df.rename(columns={"id": "scan_id"})
    log.info("extract: %d rows from %s", len(df), path)
    return df
    raise NotImplementedError

In [52]:
def check_contract(df):
    """Fail the run if the file is not the shape we agreed.

    Cheap, first, and before anything else. A missing column means nothing
    downstream can do anything sensible, so there is no reason to find out
    slowly.
    """
    missing = [c for c in REQUIRED if c not in df.columns]
    if missing:
        raise ValueError("missing columns: " + str(missing))
    if len(df) == 0:
        raise ValueError("file is empty")
    log.info("contract: ok, %d columns", len(df.columns))
    raise NotImplementedError

In [53]:
def parse_timestamps(series):
    """Two formats arrive. Return one datetime column.

    Roughly 3% of rows use DD/MM/YYYY HH:MM instead of the ISO form. The
    two-format parse in Lecture 13 is the pattern; note that the wrong format
    with errors="coerce" gives NaT rather than an exception, which is what makes
    fillna work.
    """
    iso = pd.to_datetime(series, format="ISO8601", errors="coerce")
    if iso.isna().all():
        iso = pd.to_datetime(series, format="%Y-%m-%d %H:%M:%S", errors="coerce")
    alt = pd.to_datetime(series, format="%d/%m/%Y %H:%M", errors="coerce")
    return iso.fillna(alt)
    
    raise NotImplementedError

In [54]:
def transform(raw, scan_date, hubs, couriers, customers, services):
    """Return (good, rejects).

    The decisions this function has to make, all of them visible in the data:

      * duplicate scan_id - the same event sent twice
      * timestamps in two formats
      * weight_kg with a comma decimal mark, blank, or impossible (600 kg)
      * hub_id 99, which is not a hub
      * courier_id blank, which is a real unassigned scan and not an error
      * scan_type in mixed case

    Some of those are rejects and some are repairs, and telling them apart is
    most of the work. Write the reason on every rejected row and quarantine it -
    a rejected row with no reason cannot be argued about.
    """
    df = raw.copy()
    
    for col in ["courier_id", "customer_id", "service_level_id"]:
        if col not in df.columns:
            df[col] = None

    initial_count = len(df)
    df = df.drop_duplicates(subset=["scan_id"], keep="first")
    duplicates_count = initial_count - len(df)
    df["scanned_at"] = parse_timestamps(df["scanned_at"])
    df["weight_kg"] = df["weight_kg"].astype(str).str.replace(",", ".", regex=False)
    df["weight_kg"] = pd.to_numeric(df["weight_kg"], errors="coerce")
    df["scan_type"] = df["scan_type"].str.lower().str.strip()

    df["reject_reason"] = None
    df.loc[df["scanned_at"].isna(), "reject_reason"] = "unparseable timestamp"
    df.loc[df["weight_kg"].isna() | (df["weight_kg"] <= 0) | (df["weight_kg"] > 500), "reject_reason"] = "bad weight"
    df.loc[df["hub_id"].astype(str) == "99", "reject_reason"] = "invalid hub 99"
    df.loc[~df["hub_id"].astype(str).isin(hubs["hub_id"].astype(str)), "reject_reason"] = "unknown hub"

    good = df[df["reject_reason"].isna()].copy()
    rejects = df[df["reject_reason"].notna()]

    if len(rejects) > 0:
        rejects.to_csv(f"quarantine/rejects_{scan_date}.csv", index=False)
        log.warning("quarantined %d rows to quarantine/rejects_%s.csv", len(rejects), scan_date)

    good["hub_id"] = good["hub_id"].astype(str)
    good["weight_kg"] = good["weight_kg"].round(2)
    good["scanned_at"] = good["scanned_at"]

    log.info("transform: %d read = %d good + %d rejected + %d duplicates",
             initial_count, len(good), len(rejects), duplicates_count)
    
    return good, rejects, duplicates_count
    raise NotImplementedError

In [62]:
def load(good, scan_date):
    with engine.begin() as conn:
        conn.execute(text("DELETE FROM parcel_scans WHERE DATE(scanned_at) = :d"), {"d": scan_date})
        if not good.empty:
            good.to_sql("parcel_scans", conn, if_exists="append", index=False)
    log.info("load: %d rows for %s", len(good), scan_date)
    return len(good)
    raise NotImplementedError

In [56]:
def write_load_log(scan_date, rows_read, rows_loaded, rows_rejected, duplicates_count, status, error_message=None):
    with engine.begin() as conn:
        conn.execute(text("""
            INSERT INTO load_log (scan_date, rows_read, rows_loaded, rows_rejected, rows_deduplicated, status, error_message, executed_at)
            VALUES (:scan_date, :rows_read, :rows_loaded, :rows_rejected, :rows_deduplicated, :status, :error_message, NOW())
        """), {
            "scan_date": scan_date,
            "rows_read": rows_read,
            "rows_loaded": rows_loaded,
            "rows_rejected": rows_rejected,
            "rows_deduplicated": duplicates_count,
            "status": status,
            "error_message": error_message
        })

In [57]:
def main(scan_date):
    log.info("run starting for %s", scan_date)
    try:
        raw = extract(scan_date)
        check_contract(raw)
        hubs = pd.read_csv(DATA_DIR + "/hubs.csv")
        couriers = pd.read_csv(DATA_DIR + "/couriers.csv")
        services = pd.read_csv(DATA_DIR + "/service_levels.csv")
        customers = pd.read_sql_query("SELECT customer_id FROM customers", engine)
        good, rejects = transform(raw, scan_date, hubs, couriers, customers, services)
        load(good, scan_date)
        # TODO. Write the run to load_log before returning - rows read, loaded,
        write_load_log(scan_date, rows_read, rows_loaded, rows_rejected, duplicates_count, "SUCCESS")
        log.info("run finished for %s", scan_date)
        return 0
    except Exception as problem:
        log.error("run FAILED for %s: %s", scan_date, problem)
        return 1

In [71]:
main("2026-05-04")


2026-09-15 21:40:38,872 INFO run starting for 2026-05-04
2026-09-15 21:40:38,879 INFO extract: 1285 rows from ../data\scans_2026-05-04.csv
2026-09-15 21:40:38,880 INFO contract: ok, 9 columns
2026-09-15 21:40:38,881 ERROR run FAILED for 2026-05-04: 


1

In [72]:
if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--date", required=True)
    sys.exit(main(parser.parse_args().date))

usage: ipykernel_launcher.py [-h] --date DATE
ipykernel_launcher.py: error: the following arguments are required: --date


SystemExit: 2

C:\Users\omar abo khadair\AppData\Roaming\Python\Python39\site-packages\IPython\core\interactiveshell.py:3558: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
